## Unit 05. SciPy 特殊函式 課堂作業
- 課程名稱: 電腦在化工上之應用 (ChemE 3502)
- 學年度: 114下
- 上課時間: 每週一 16:10-19:20
-
- 指導教授: 莊曜禎 助理教授
- 學生姓名: ＯＯＯ
- 學號: B12345678
- email帳號: fcu.b12345678@gmail.com

---
### 0. 環境設定

In [1]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit05_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit05'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: d:\MyGit\ChemE-3502\Unit05
✓ 結果輸出目錄: d:\MyGit\ChemE-3502\Unit05\outputs\Unit05_Homework
✓ 圖檔輸出目錄: d:\MyGit\ChemE-3502\Unit05\outputs\Unit05_Homework\figs


---
### 1. 載入套件

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy      版本: {np.__version__}")
import scipy
print(f"  scipy      版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

✓ 套件載入完成
  numpy      版本: 1.23.5
  scipy      版本: 1.15.2
  matplotlib 版本: 3.10.8


---
## I. 練習題一：Gamma 函式與不完全 Gamma 函式 — 反應器停留時間分佈分析

**背景說明**

在連續攪拌槽反應器（CSTR）串聯系統中，系統的停留時間分佈（Residence Time Distribution, RTD）可用 **Gamma 分佈** 描述：

$$
E(t) = \frac{1}{\theta^k \Gamma(k)} t^{k-1} e^{-t/\theta}
$$

其中 $k$ 為串聯槽數（形狀參數）， $\theta$ 為每個槽的平均停留時間（尺度參數），平均停留時間 $\bar{t} = k\theta$，變異數 $\sigma^2 = k\theta^2$。

累積 RTD 函式 $F(t)$ 定義為：

$$
F(t) = \int_0^t E(t')\,dt' = I\!\left(k,\, \frac{t}{\theta}\right)
$$

其中 $I(k, x) = $ `scipy.special.gammainc(k, x)` 為**正規化不完全 Gamma 函式**，`scipy.special.gammaincc(k, x)` 為其補函式（存活函式 $S(t)=1-F(t)$）。

---

**作業要求**

(1) **定義兩種反應器配置**：
- 配置 A：$k=2, \theta=3\,\text{min}$（2 槽 CSTR，每槽均等）
- 配置 B：$k=5, \theta=1.5\,\text{min}$（5 槽 CSTR，每槽均等）

(2) **計算並輸出**下列時間點的累積 RTD $F(t)$ 與存活函式 $S(t)$：
- $t = [2, 5, 8, 10, 15]\,\text{min}$（兩種配置均需計算）

(3) **求分位數時間**：使用 `special.gammaincinv(k, p)` 計算兩種配置的 $t_{10\%}$、$t_{50\%}$、$t_{90\%}$（即 $F(t)=0.1,0.5,0.9$ 對應的時間）

(4) **繪製圖表**：
- 左圖：$E(t)$ 停留時間分佈密度函式（兩種配置疊圖）
- 右圖：$F(t)$ 累積分佈函式 + 三條水平輔助線（$F=0.1, 0.5, 0.9$）

(5) **儲存圖檔** 至 `FIG_DIR / 'hw1_rtd_gamma.png'`

**提示**：$E(t)$ 可由 $F(t)$ 差分近似，或直接使用 Gamma 分佈 PDF 公式計算

In [ ]:
# ================================================================
# 練習一：Gamma 函式與不完全 Gamma 函式 — 反應器停留時間分佈分析
# ================================================================
# 請在此編寫你的程式碼
# 提示：
#   special.gammainc(k, t/theta)      → F(t) 累積 RTD
#   special.gammaincc(k, t/theta)     → S(t) 存活函式
#   special.gammaincinv(k, p)         → 給定機率 p 反求 t/theta

# (1) 定義參數
# ...

# (2) 計算 F(t) 與 S(t) 於指定時間點
# ...

# (3) 計算分位數時間 t_10%, t_50%, t_90%
# ...

# (4) 繪圖
# ...

# (5) 儲存圖檔
# ...


---
## II. 練習題二：誤差函式 — 非穩態擴散質傳（滲透理論）

**背景說明**

**滲透理論（Penetration Theory）**描述氣體溶解至液體的非穩態質傳過程。半無限液膜模型（$0 < x < \infty$）中，質傳 PDE 的解析解為：

$$
\frac{C(x,t) - C_0}{C_s - C_0} = \mathrm{erfc}\!\left(\frac{x}{2\sqrt{D t}}\right)
$$

其中 $C_s$ 為界面平衡濃度（mol/L），$C_0$ 為初始液相濃度，$D$ 為擴散係數（m²/s），$x$ 為深度（m），$t$ 為接觸時間（s）。

瞬間質傳通量 $N_A(t)$ 與總吸收量 $Q(t)$：

$$
N_A(t) = (C_s - C_0)\sqrt{\frac{D}{\pi t}}, \qquad Q(t) = 2(C_s - C_0)\sqrt{\frac{D t}{\pi}}
$$

當液膜厚度 $\delta$ 有限時（液膜理論），也需要 `erf` / `erfc` 計算誤差補函式。

---

**作業要求**

(1) **定義參數**：
- $C_s = 0.05\,\text{mol/L}$，$C_0 = 0\,\text{mol/L}$
- $D = 2.0 \times 10^{-9}\,\text{m}^2/\text{s}$（氧氣在水中的擴散係數）
- 深度範圍 $x = [0, 0.1, 0.5, 1.0, 2.0, 5.0]\,\text{mm}$（轉換為 m）
- 時間 $t = [0.01, 0.1, 1.0, 10.0]\,\text{s}$

(2) **建立濃度剖面表格**：計算各 $(x, t)$ 組合的 $C(x,t)$ 並輸出為二維表格（行為深度，列為時間）

(3) **計算並輸出**：
- 各時間點的瞬間質傳通量 $N_A(t)$（mol/m²/s）
- 各時間點的累積吸收量 $Q(t)$（mol/m²）
- 驗證數值穩定性：比較 `special.erfc(z)` 與 `1 - special.erf(z)` 在大 $z$ 值（如 $z=3,4,5$）時的差異

(4) **繪製圖表**：
- 左圖：4 條不同時間的濃度剖面 $C(x,t)$ vs $x$
- 右圖：$N_A(t)$ 與 $Q(t)$ vs $t$（雙 Y 軸）

(5) **儲存圖檔** 至 `FIG_DIR / 'hw2_penetration_erf.png'`

**提示**：`special.erfc(x/(2*np.sqrt(D*t)))` 計算無量綱濃度；
注意單位換算（mm → m）；`special.erfcx(z)` 可避免大 $z$ 下溢位

In [ ]:
# ================================================================
# 練習二：誤差函式 — 非穩態擴散質傳（滲透理論）
# ================================================================
# 請在此編寫你的程式碼
# 提示：
#   special.erfc(x / (2*np.sqrt(D*t)))   → 無量綱濃度
#   special.erfcx(z)                      → 大 z 值數值穩定版本
#   C_s * special.erfc(z)                 → 實際濃度 C(x,t)

# (1) 定義參數
# ...

# (2) 建立濃度剖面表格
# ...

# (3) 計算 NA(t), Q(t) 並驗證 erfc vs 1-erf 精度差異
# ...

# (4) 繪圖
# ...

# (5) 儲存圖檔
# ...


---
## III. 練習題三：修正 Bessel 函式 — 圓柱形觸媒粒子有效因子

**背景說明**

對圓柱形觸媒粒子（無限長圓柱），有效因子 $\eta$ 由 **Thiele 模數** $\phi$ 決定：

$$
\eta = \frac{2}{\phi} \cdot \frac{I_1(\phi)}{I_0(\phi)}
$$

其中 $I_0, I_1$ 為**第一類修正 Bessel 函式**（`special.iv(0, phi)`, `special.iv(1, phi)`）。

球形觸媒粒子的有效因子：

$$
\eta_{\text{sphere}} = \frac{3}{\phi^2}\!\left[\phi \coth(\phi) - 1\right] = \frac{3}{\phi}\!\left[\frac{1}{\tanh(\phi)} - \frac{1}{\phi}\right]
$$

平板觸媒粒子的有效因子：

$$
\eta_{\text{flat}} = \frac{\tanh(\phi)}{\phi}
$$

Thiele 模數定義：

$$
\phi = L \sqrt{\frac{k_s}{D_e}}
$$

其中 $L$ 為特徵長度，$k_s$ 為表面反應速率常數（s⁻¹），$D_e$ 為有效擴散係數（m²/s）。

---

**作業要求**

(1) **計算三種幾何形狀**（圓柱/球形/平板）在以下 Thiele 模數的有效因子：
- $\phi = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]$

(2) **輸出比較表格**：每行為一個 $\phi$ 值，三欄分別為三種幾何的 $\eta$

(3) **找出擴散限制閾值**：各幾何形狀中， $\eta < 0.95$ 的最小 $\phi$ 值（插值或數值求解）

(4) **反應器設計問題**：已知圓柱觸媒粒子半徑 $R = [0.5, 1.0, 2.0, 5.0]\,\text{mm}$，$k_s = 0.1\,\text{s}^{-1}$，$D_e = 1 \times 10^{-9}\,\text{m}^2/\text{s}$，計算每種粒徑的 Thiele 模數與有效因子，說明哪種粒徑能維持 $\eta > 0.9$

(5) **繪製** $\eta$ vs $\phi$（log scale）並標記三條曲線，儲存至 `FIG_DIR / 'hw3_bessel_effectiveness.png'`

**提示**：
- 圓柱：`2/phi * special.iv(1, phi) / special.iv(0, phi)`
- 球形：`(3/phi) * (1/np.tanh(phi) - 1/phi)`
- 注意 $\phi \to 0$ 時的極限處理（可用 `np.where` 避免除零）

In [ ]:
# ================================================================
# 練習三：修正 Bessel 函式 — 圓柱形觸媒粒子有效因子
# ================================================================
# 請在此編寫你的程式碼
# 提示：
#   special.iv(0, phi)   → I_0(phi)
#   special.iv(1, phi)   → I_1(phi)
#   eta_cyl = 2/phi * special.iv(1, phi) / special.iv(0, phi)
#   eta_sph = (3/phi) * (1/np.tanh(phi) - 1/phi)   (phi > 0)
#   eta_flat = np.tanh(phi) / phi                    (phi > 0)

# (1) 定義 Thiele 模數陣列與計算三種有效因子
# ...

# (2) 輸出比較表格
# ...

# (3) 找出 η < 0.95 的閾值
# ...

# (4) 反應器設計問題：計算不同粒徑的 η
# ...

# (5) 繪圖並儲存
# ...


---
## IV. 練習題四：Lambert W 函式 — Langmuir-Hinshelwood 反應動力學封閉解

**背景說明**

**Langmuir-Hinshelwood（LH）動力學**常見於觸媒反應，速率式為：

$$
r_A = \frac{k \cdot K_A \cdot C_A}{1 + K_A \cdot C_A}
$$

對批次反應器（Batch Reactor），質量守恆給出：

$$
\frac{dC_A}{dt} = -\frac{k \cdot K_A \cdot C_A}{1 + K_A \cdot C_A}
$$

此 ODE 可**解析求解**，解為含 Lambert W 函式的封閉形式：

$$
C_A + \frac{1}{K_A} \ln C_A = C_{A0} + \frac{1}{K_A} \ln C_{A0} - k\,t
$$

整理後：

$$
C_A(t) = \frac{1}{K_A}\,W\!\left(K_A \cdot C_{A0} \cdot e^{K_A \left(C_{A0} - k\,t\right)}\right)
$$

其中 $W(\cdot)$ 為 Lambert W 函式（主值分支）： `special.lambertw(x).real`。

---

**作業要求**

(1) **定義動力學參數**：
- $k = 0.05\,\text{mol/(L} \cdot \text{min)}$（最大反應速率）
- $K_A = 2.0\,\text{L/mol}$（吸附平衡常數）
- $C_{A0} = 1.0\,\text{mol/L}$（初始濃度）
- 時間範圍：$t = [0, 2, 4, 6, 8, 10, 15, 20]\,\text{min}$

(2) **計算解析解** $C_A(t)$ 並驗證：將 $C_A(t)$ 代回 ODE 右側，確認 $dC/dt$ 吻合有限差分估計值

(3) **比較三種條件**（$K_A = 0.5, 2.0, 10.0\,\text{L/mol}$，固定 $k=0.05$）的轉化率曲線 $X_A = 1 - C_A/C_{A0}$

(4) **求 50% 轉化率時間**：使用解析解，對三種 $K_A$ 分別求 $X_A=0.5$ 對應的時間（解方程式 $C_A = 0.5\,C_{A0}$）

(5) **繪製** $C_A(t)$ 與 $X_A(t)$ 圖並標記各曲線，儲存至 `FIG_DIR / 'hw4_lambertw_kinetics.png'`

**提示**：
- `special.lambertw(x).real` 取實部（主值分支 $W_0$）
- Lambert W 的引數 $= K_A \cdot C_{A0} \cdot \exp(K_A \cdot (C_{A0} - k \cdot t))$
- 驗證 $W(x) \cdot e^{W(x)} = x$（Lambert W 的定義屬性）

In [ ]:
# ================================================================
# 練習四：Lambert W 函式 — LH 反應動力學封閉解
# ================================================================
# 請在此編寫你的程式碼
# 提示：
#   W_arg = K_A * C_A0 * np.exp(K_A * (C_A0 - k * t))
#   C_A   = special.lambertw(W_arg).real / K_A
#   驗證: W(x) * exp(W(x)) == x?

# (1) 定義動力學參數
# ...

# (2) 計算解析解 C_A(t) 並驗證 ODE 殘差
# ...

# (3) 比較三種 K_A 條件
# ...

# (4) 求 50% 轉化率時間
# ...

# (5) 繪圖並儲存
# ...


---
## V. 練習題五：正態 CDF/分位數函式 — 製程統計品質管制 (SPC)

**背景說明**

製程能力指標 **$C_{pk}$** 量化製程能力與規格寬度的比值：

$$
C_{pk} = \frac{\min(\mu - LSL,\, USL - \mu)}{3\sigma}
$$

對應的**管制帶寬**（sigma 倍數）：$z = 3 C_{pk}$。

雙尾**不良品率（PPM）**：

$$
\text{PPM} = 2 \times (1 - \Phi(z)) \times 10^6 = \mathrm{erfc}\!\left(\frac{z}{\sqrt{2}}\right) \times 10^6
$$

其中 $\Phi(z) = $ `special.ndtr(z)` 為標準正態 CDF，反函式 `special.ndtri(p)` 可由信賴水準反推 $z$ 閾值。

在製程警報系統設計中，給定警報虛驚率（False Alarm Rate）$\alpha$，警報閾值為：

$$
h = \Phi^{-1}(1 - \alpha/2) \cdot \sigma
$$

---

**作業要求**

(1) **計算並輸出製程能力表格**，$C_{pk} = [0.67, 1.00, 1.33, 1.50, 1.67, 2.00]$：
- 每列輸出：$C_{pk}$、$z = 3C_{pk}$、雙尾 PPM、單側 PPM

(2) **反向設計**：若品質要求 PPM $\leq [1000, 100, 10, 1, 0.1]$，使用 `special.ndtri` 計算各需求對應的最小 $C_{pk}$

(3) **警報系統設計**：給定製程 $\mu = 100, \sigma = 2$，規格 $[LSL=94, USL=106]$，
- 計算 $C_{pk}$
- 設計 $1\sigma, 2\sigma, 3\sigma$ 警報帶，並計算各帶的虛驚率（FAR）與年均虛驚次數（假設每分鐘取樣一次，一年 525600 分鐘）

(4) **Beta 分佈應用（進階）**：若品質工程師對製程偏移有 Beta 先驗 $\text{Beta}(4, 6)$，使用 `special.betainc` 與 `special.betaincinv` 計算：
- $P[\eta \leq 0.5]$（偏移比例低於 0.5 的機率）
- 90% 信賴上限（$\alpha = 0.9$ 時的分位數）

(5) **繪製圖表**：
- 左圖：$C_{pk}$ vs PPM（對數尺度 Y 軸）
- 右圖：正態分佈 PDF + 規格限線 + 警報帶（$1\sigma, 2\sigma, 3\sigma$ 不同顏色填充）
- 儲存至 `FIG_DIR / 'hw5_spc_ndtr.png'`

**提示**：
- `ppm = 2 * (1 - special.ndtr(3 * Cpk)) * 1e6`
- `special.ndtri(1 - alpha/2)` 求警報閾值
- `special.betainc(a, b, x)` 求 Beta CDF

In [ ]:
# ================================================================
# 練習五：正態 CDF/分位數 — 製程統計品質管制 (SPC)
# ================================================================
# 請在此編寫你的程式碼
# 提示：
#   special.ndtr(z)         → Φ(z) 標準正態 CDF
#   special.ndtri(p)        → Φ^{-1}(p) 標準正態分位數
#   special.erfc(z/np.sqrt(2)) → 雙尾尾端機率 (更精確)
#   special.betainc(a, b, x)   → Beta CDF
#   special.betaincinv(a, b, p) → Beta 分位數

# (1) 計算製程能力表格
# ...

# (2) 反向設計：由 PPM 需求求最小 Cpk
# ...

# (3) 警報系統設計：計算虛驚率與年均虛驚次數
# ...

# (4) Beta 先驗分佈計算
# ...

# (5) 繪圖並儲存
# ...


---
## II. 額外加分作業（自由繳交）

- 學生可自由選擇是否繳交加分作業（下週上課前完成 email 電子檔）
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分
- 分數加的不多，真的不一定要交加分作業，正常出席上課做好期末報告即可
- 加分作業不一定要全部都做完，想繳交至少完成其中一項即可
- 請務必自行完成！你可以問 AI、問同學、問學長姊，但一定要自行**吸收**、**執行**、**整理**結果！
- 不要貼 AI 的答案寄給老師看，你自己看就好
- 如果系統自動比對發現作業內容（與他人雷同、原創性低、抄襲比例過高），作業分數會 0 分
- 如果你直接 100% 複製別人的作業繳交，你會直接被當掉（提供作業給別人複製的也一樣）
- 老師看你作業要花時間，真的有心想寫加分作業，請好好自己做

---
### 加分實驗一：`logsumexp` 與 Boltzmann 統計 — 化學反應平衡常數溫度依賴性

**背景**：反應 $\mathrm{A \rightleftharpoons B + C}$ 的各分子構型 Boltzmann 分佈，利用 `logsumexp` 計算低溫/高溫下的配分函數比值，分析溫度對平衡的影響。

**要求**：
1. 設定 A、B、C 三分子各 4 個能態（自行設定能量差 $\Delta E$）
2. 計算溫度範圍 $T = [300, 500, 700, 1000, 1500]\,\text{K}$ 下的各分子配分函數 $Z_A, Z_B, Z_C$
3. 計算熱力學平衡常數（配分函數比） $K(T) = Z_B \cdot Z_C / Z_A$
4. 繪製 $\ln K$ vs $1/T$（van't Hoff plot），由斜率估算 $\Delta H$ 與 $\Delta S$
5. 儲存圖檔至 `FIG_DIR / 'bonus1_logsumexp_equilibrium.png'`

In [ ]:
# ================================================================
# 加分實驗一：logsumexp — 配分函數與化學反應平衡常數
# ================================================================
# 請在此編寫你的程式碼



---
### 加分實驗二：`psi` / `polygamma` — Gamma 分佈 MLE 參數估計（反應器資料分析）

**背景**：批次反應器的操作時間（停留時間）假設服從 Gamma 分佈，利用最大似然估計（MLE）從樣本數據估計形狀參數 $\hat{a}$。

MLE 方程需要 Newton-Raphson 法求解，每步迭代用到：
- `special.psi(a)` — $\psi(a) = d\ln\Gamma(a)/da$（MLE 方程的梯度）
- `special.polygamma(1, a)` — $\psi'(a)$（Newton step 的 Hessian）

**要求**：
1. 使用 `np.random.seed(42)` 生成 200 個 $\text{Gamma}(a=3.0, \theta=2.0)$ 的樣本（停留時間數據）
2. 計算 $\bar{x}$ 與 $\overline{\ln x}$（樣本統計量）
3. 實作 Newton-Raphson 迭代求解 MLE 方程：$\psi(\hat{a}) - \ln(\hat{a}) = \overline{\ln x} - \ln\bar{x}$，顯示每步迭代的 $\hat{a}$ 收斂過程
4. 繪製樣本直方圖 + 擬合 Gamma 分佈 PDF
5. 儲存圖檔至 `FIG_DIR / 'bonus2_psi_gamma_mle.png'`

In [ ]:
# ================================================================
# 加分實驗二：psi / polygamma — Gamma MLE Newton-Raphson 迭代
# ================================================================
# 請在此編寫你的程式碼



---
## 💭 思考題

請以文字（1-3 句話）回答下列問題，寫在此 cell 下方或另開 Markdown cell：

1. **練習一**：若增加串聯槽數 $k$（固定平均停留時間 $\bar{t} = k\theta$），RTD 曲線 $E(t)$ 形狀如何變化？極限 $k \to \infty$ 代表什麼物理意義？

2. **練習二**：為什麼在計算尾端機率時（如 $z > 3$），`erfc(z)` 比 `1 - erf(z)` 更準確？這在化工設計中有什麼實際影響？

3. **練習三**：Thiele 模數 $\phi$ 增大時，有效因子 $\eta$ 下降，反應速率受擴散控制。在工業觸媒設計中，如何在**觸媒活性**與**擴散效率**之間取得平衡？

4. **練習四**：Lambert W 函式提供了 LH 動力學的解析解，相比數值 ODE 求解（如 `scipy.integrate.solve_ivp`），解析解有哪些優點與限制？

5. **練習五**：$C_{pk}$ 從 1.33 提升到 1.67 時，PPM 從約 66 降到約 0.5，提升幅度極大。但在化工實務中，追求過高製程能力指數是否永遠合理？請從成本與風險的角度討論。

**你的回答：**

1. （在此填寫你對問題一的回答）

2. （在此填寫你對問題二的回答）

3. （在此填寫你對問題三的回答）

4. （在此填寫你對問題四的回答）

5. （在此填寫你對問題五的回答）

---
# 想對老師說的話

（在此填寫你想對老師說的話）